# GRU Implement from Scratch

# Import Libraries

In [26]:
import numpy as np

In [27]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Input

# Custom `GRU`

✅ **1. Combined weight version (your version)**

You treat `[h_{t-1}, x_t]` as one long input vector.
So:

* If hidden size = H
* If input size = I
* Then combined vector size = **H + I**

Meaning:

$$
W_r \in \mathbb{R}^{H \times (H+I)}
$$

$$
W_z \in \mathbb{R}^{H \times (H+I)}
$$

This version uses **ONE big weight matrix per gate**.

---

✅ **2. Split-weight version (most textbooks use this)**

The combined matrix is split into:

* Part that multiplies the input → (W_r)
* Part that multiplies hidden state → (U_r)

So:

$$
r_t = \sigma(W_r x_t + U_r h_{t-1} + b_r)
$$

$$
z_t = \sigma(W_z x_t + U_z h_{t-1} + b_z)
$$

This version is mathematically **identical** to the combined version.

---

🎯 **How the two versions match**

### Combined version:

$$
W_r \cdot [h_{t-1}, x_t]
$$

Let’s split ( W_r ) into two blocks:

$$
W_r = [U_r ;|; W_r']
$$

Then:

$$
W_r \cdot [h_{t-1}, x_t]
= U_r h_{t-1} + W_r' x_t
$$

So we simply rename:

$$
W_r' = W_r
$$

And we get:

$$
r_t = \sigma(U_r h_{t-1} + W_r x_t)
$$

Which is exactly the standard form used in deep learning frameworks.

---

⭐ **So the correct rewritten forms are:**

### **Reset gate**

$$
r_t = \sigma(W_r x_t + U_r h_{t-1} + b_r)
$$

### **Update gate**

$$
z_t = \sigma(W_z x_t + U_z h_{t-1} + b_z)
$$

In [28]:
class CustomGRU:
  def __init__(self, input_size, hidden_size):
      self.hidden_size = hidden_size

      # weights for r gate
      self.W_r = np.random.randn(hidden_size, input_size)
      self.U_r = np.random.randn(hidden_size, hidden_size)
      self.b_r = np.zeros((hidden_size, 1))

      # weights for z gate
      self.W_z = np.random.randn(hidden_size, input_size)
      self.U_z = np.random.randn(hidden_size, hidden_size)
      self.b_z = np.zeros((hidden_size, 1))

      # weights for candidate h~
      self.W_h = np.random.randn(hidden_size, input_size)
      self.U_h = np.random.randn(hidden_size, hidden_size)
      self.b_h = np.zeros((hidden_size, 1))

      # initial hidden state
      self.h = np.zeros((hidden_size, 1))

  def sigmoid(self, x):
    return 1 / (1 + np.exp(-x))

  def step(self, x):
      """
      x: input at time t (shape: input_size x 1)
      returns: new hidden state h_t
      """

      # compute gates
      r_t = self.sigmoid(self.W_r @ x + self.U_r @ self.h + self.b_r)
      z_t = self.sigmoid(self.W_z @ x + self.U_z @ self.h + self.b_z)

      # candidate hidden state
      h_tilde = np.tanh(self.W_h @ x + self.U_h @ (r_t * self.h) + self.b_h)

      # final hidden state
      self.h = (1 - z_t) * self.h + z_t * h_tilde

      return self.h

In [29]:
# Example usage
gru = CustomGRU(input_size=3, hidden_size=2)
x_t = np.random.randn(3, 1)  # one input at time t
h_t = gru.step(x_t)

print("r_t, z_t, h~_t, h_t are computed inside the step() function.")
print('Input x_t:')
print(x_t)
print('------------------')
print("Output h_t:\n", h_t)

r_t, z_t, h~_t, h_t are computed inside the step() function.
Input x_t:
[[ 0.26914933]
 [ 0.48235377]
 [-0.2048159 ]]
------------------
Output h_t:
 [[0.1984935 ]
 [0.16881292]]


# Inbuild `GRU`

In [30]:
model = Sequential([
    Input(shape=(10, 5)), # sequence length =10, features=5
    GRU(32),
    Dense(1)
])

model.compile(optimizer="adam", loss="mse")
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_2 (GRU)                     │ (None, 32)             │         3,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,777 (14.75 KB)

 Trainable params: 3,777 (14.75 KB)

 Non-trainable params: 0 (0.00 B)